# An implementation of SPViT
### ``Splitting a ViT dynamically over N decives for optimzal inference time`` 

In [3]:
import torch
import torchvision.models as models

# Load a standard ViT
model = models.vit_b_16(weights='DEFAULT')

### ``KEY POINTS:``
- A block or ``model.encoder.layers[x]`` is singular transformer encoder block 
    - It includes the self-attention module + MLP. 
    - This block is stackable hence ``model.encoder.layers[0]`` is below  ``model.encoder.layers[1]`` and so on.
- Self attention's **``in_proj_weight``** is a single representation for the QKV weights.
    - ``q_weight = attn_module.in_proj_weight[:embed_dim, :]``
    - ``k_weight = attn_module.in_proj_weight[embed_dim:embed_dim*2, :]``
    - ``v_weight = attn_module.in_proj_weight[embed_dim*2:, :]``

In [9]:
# Look at the first block
first_block = model.encoder.layers[0]
print(f"Attention Module: {first_block.self_attention}")
print(f"MLP Module: {first_block.mlp}")

# Check the shape of the Attention weights
# Typically [3 * dim, dim] because Q, K, and V are fused
print(f"QKV Weight Shape: {first_block.self_attention.in_proj_weight.shape}")

embed_dim = first_block.self_attention.in_proj_weight.shape[1]
# print("Q_weight: ", first_block.self_attention.in_proj_weight[:embed_dim, :], '\n')
# print("K_weight: ", first_block.self_attention.in_proj_weight[embed_dim:embed_dim*2, :], '\n')
# print("V_weight: ", first_block.self_attention.in_proj_weight[embed_dim*2:, :], '\n')

print(f"Query Weight Shape: {first_block.self_attention.in_proj_weight[:embed_dim, :].shape}") # [768, 768]

Attention Module: MultiheadAttention(
  (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
)
MLP Module: MLPBlock(
  (0): Linear(in_features=768, out_features=3072, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.0, inplace=False)
  (3): Linear(in_features=3072, out_features=768, bias=True)
  (4): Dropout(p=0.0, inplace=False)
)
QKV Weight Shape: torch.Size([2304, 768])
Query Weight Shape: torch.Size([768, 768])


## ``What and How to Divide?``
- The Head-Width Splitting
    - Divide heads into N devices based on how much work a node in a cluster can do

- The Output Neuron Split
    - Divide the output layer neurons amongst Nodes in a cluster

``NOTE:`` After the computation everything is concatenated before sending to the next layer or block

## ``HEAD-Width Splitting``

In [31]:
# model configuration
embed_dim = first_block.self_attention.in_proj_weight.shape[1]
num_heads = first_block.self_attention.num_heads
head_dim = embed_dim // num_heads 

print('-'*75, f"Embed Dim: {embed_dim}\t|\tNum of heads: {num_heads}\t|\tHead dimension: {head_dim}", '-'*75, sep='\n')

# 1. Get the original weights
attn_module = model.encoder.layers[0].self_attention
in_proj_weight = attn_module.in_proj_weight # [2304, 768]

# 2. Define our Split (Edge gets 25% of work, GPU gets 75% of work)
RATIO = 0.25
edge_heads_total = int(num_heads * RATIO)
# gpu_heads_total  = num_heads - edge_heads_total

edge_heads = range(0, edge_heads_total)
gpu_heads  = range(edge_heads_total, num_heads)

def get_head_weights(full_weight, head_indices, embed_dim, head_dim):
    # We need to pick the rows for Q, K, and V for the specific heads
    # Q is 0:768, K is 768:1536, V is 1536:2304  => (refer to key notes in first markdown)
    # Essentially this function extract specific rows of the combined QKV weights that are for specific head using simple offset maths and returns a concatenated weight matrix
    slices = []
    for i in [0, 1, 2]: # Q, K, V
        offset = i * embed_dim
        for h in head_indices:
            start = offset + (h * head_dim)
            end = start + head_dim
            slices.append(full_weight[start:end, :])
    
    return torch.cat(slices, dim=0)

# Extract partial weights
edge_w = get_head_weights(in_proj_weight, edge_heads, embed_dim, head_dim) # [3*192, 768]
gpu_w  = get_head_weights(in_proj_weight, gpu_heads,  embed_dim, head_dim) # [3*576, 768]

print(f"Edge Weight Shape: {edge_w.shape}")
print(f"GPU Weight Shape: {gpu_w.shape}")


"""This Allows to divide matmul ops to different divices. Now we only need to know a dynamic calculation of percentage of work for each node (we'll do this using the ARIMA-V algorithm)"""

---------------------------------------------------------------------------
Embed Dim: 768	|	Num of heads: 12	|	Head dimension: 64
---------------------------------------------------------------------------
Edge Weight Shape: torch.Size([576, 768])
GPU Weight Shape: torch.Size([1728, 768])


"This Allows to divide matmul ops to different divices. Now we only need to know a dynamic calculation of percentage of work for each node (we'll do this using the ARIMA-V algorithm)"

### ``Dummy forward pass through the attention block``

In [44]:
def merge_n_projections(projections):
    """
    Merges QKV projections from N different devices.
    
    Args:
        projections (list): List of tensors from N devices. 
                           Each has shape [Batch, Tokens, 3 * partial_head_dim]
    """
    all_q = []
    all_k = []
    all_v = []
    
    for p in projections:
        # Each device calculated its own Q, K, and V
        # chunk(3, dim=-1) splits [..., 3*N] into three [..., N] tensors
        q, k, v = torch.chunk(p, 3, dim=-1)
        
        all_q.append(q)
        all_k.append(k)
        all_v.append(v)
    
    # Re-group: All Qs from all devices, then all Ks, then all Vs
    q_final = torch.cat(all_q, dim=-1)
    k_final = torch.cat(all_k, dim=-1)
    v_final = torch.cat(all_v, dim=-1)
    
    # Final assembly into the [Batch, Tokens, 3 * Embed_Dim] format
    return torch.cat([q_final, k_final, v_final], dim=-1)

# 1. Setup Dummy Data
# ViT input shape is usually [Batch, Tokens, Embed_Dim]
# e.g., 1 image, 197 tokens (14x14 patches + 1 class token), 768 dims
X = torch.randn(1, 197, 768)

# 2. Compute Edge Projection (Heads 0-2)
# weight.T is used because Linear layers in PyTorch are [out_features, in_features]
edge_proj = X @ edge_w.t() # Result shape: [1, 197, 192] (3 heads * 64)

# 3. Compute GPU Projection (Heads 3-11)
gpu_proj = X @ gpu_w.t()   # Result shape: [1, 197, 576] (9 heads * 64)

# 4. The Merge (Concatenate back to 768)
all_projections = [edge_proj, gpu_proj]
merged_proj = merge_n_projections(all_projections)

# 5. The Verification
# Let's compare it to the original model's projection
# Note: original in_proj doesn't include the bias for this test, 
# but we can check against the raw weight multiplication
with torch.no_grad():
    original_proj = X @ in_proj_weight.t() # .t() is transpose

# Check if they are identical
difference = torch.abs(merged_proj - original_proj).max()
print(f"Max Difference: {difference.item()}") # difference is an object .item() is simply for extraction of value
# This should be 0.0 (or a very tiny number due to float precision) which proves everything is working correctly

Max Difference: 0.0


## ``Output Neuron Splitting``

ONS treats two layers as a single fused operation to avoid talking over the network.

- The Mathematical View: The "Inside-Out" Split
    - Think of the MLP as a sandwich:
        - Layer 1 (Expansion): Takes 768 → 3072.
        - Layer 2 (Projection): Takes 3072 → 768.
- When you split the rows of Layer 1, you are deciding which "intermediate neurons" a device is responsible for. If the Edge device takes rows 0–1000, it is the only one that knows how to calculate the values for those specific 1000 hidden neurons.
- Now, look at Layer 2. Its job is to take those 3072 neurons and turn them back into 768. If the Edge device only has the values for neurons 0–1000, it can only use the columns of Layer 2 that correspond to those neurons.
- In short:
    - Layer 1 Row Split: "I am only going to create these 1000 neurons."
    - Layer 2 Column Split: "I am only going to use these 1000 neurons to contribute to the final 768."

- As a result we skip the concat after layer 1 and only concat after layer 2 (the whole output of the encoder block)

In [ ]:
mlp_module = model.encoder.layers[0].mlp # first block's MLP part

print(mlp_module)

w1 = mlp_module[0].weight # [3072, 768]
w2 = mlp_module[3].weight # [768, 3072]

# Split point (e.g., first 1000 neurons)
split = 1000

# Edge Weights
edge_w1 = w1[:split, :]    # [1000, 768]
edge_w2 = w2[:, :split]    # [768, 1000] <-- Note we slice the columns here!

# GPU Weights
gpu_w1 = w1[split:, :]     # [2072, 768]
gpu_w2 = w2[:, split:]     # [768, 2072]

MLPBlock(
  (0): Linear(in_features=768, out_features=3072, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.0, inplace=False)
  (3): Linear(in_features=3072, out_features=768, bias=True)
  (4): Dropout(p=0.0, inplace=False)
)


# PUTTING IT ALL TOGETHER

### After this the MLP portion of the encoder is left

In [50]:
def get_model_metadata(model):
    # 1. Get Embedding Dimension from the first layer's weights
    embed_dim = model.encoder.layers[0].self_attention.in_proj_weight.shape[1]
    
    # 2. Get Sequence Length (number of tokens)
    # This is stored in the positional embedding: [1, Seq_Len, Embed_Dim]
    seq_length = model.encoder.pos_embedding.shape[1]
    
    # 3. Get MLP Hidden Dimension
    # Typically 4x the embed_dim (3072 for Base)
    mlp_hidden_dim = model.encoder.layers[0].mlp[0].weight.shape[0]
    
    # 4. Get Number of Heads
    num_heads = model.encoder.layers[0].self_attention.num_heads
    
    return {
        "embed_dim": embed_dim,
        "seq_length": seq_length,
        "mlp_hidden_dim": mlp_hidden_dim,
        "num_heads": num_heads
    }

def ready_for_math(t, meta):
    return t.view(1, meta["seq_length"], meta["num_heads"], head_dim).transpose(1, 2)


In [ ]:

import torch
import torchvision.models as models

# Load model
model = models.vit_b_16(weights='DEFAULT')
layers = model.encoder.layers
meta = get_model_metadata(model)

# Setup dynamic data
X = torch.randn(1, meta["seq_length"], meta["embed_dim"])
current_state = X

# ARIMA-V Ratio (Calculated once or per block)
RATIO = 0.4  # Edge does 40% of heads/neurons
head_dim = meta["embed_dim"] // meta["num_heads"]

for i, block in enumerate(model.encoder.layers):
    # --- STEP 1: RESIDUAL 1 (Attention) ---
    identity = current_state
    x = block.ln_1(current_state)
    
    # Calculate Head Slices
    edge_h_count = int(meta["num_heads"] * RATIO)
    edge_heads = range(0, edge_h_count)
    gpu_heads = range(edge_h_count, meta["num_heads"])
    
    # Extract Weights Dynamically
    edge_attn_w = get_head_weights(block.self_attention.in_proj_weight, edge_heads, meta["embed_dim"], head_dim)
    gpu_attn_w  = get_head_weights(block.self_attention.in_proj_weight, gpu_heads, meta["embed_dim"], head_dim)
    
    # Parallel Projections
    edge_qkv = x @ edge_attn_w.t()
    gpu_qkv = x @ gpu_attn_w.t()
    
    # Merge & finish Attention (Conceptual merge)
    merged_qkv = merge_n_projections([edge_qkv, gpu_qkv])
    
    # To keep the simulation running, we use the internal forward for the 'softmax' part
    # but the logic above proves the data split works!
    # attn_out, _ = block.self_attention(x, x, x, need_weights=False)
    # current_state = identity + attn_out
    
    # 2. Add the BIAS (Very important! merge_n_projections only did weights)
    # We must split and merge the bias just like the weights
    in_proj_bias = block.self_attention.in_proj_bias
    # (Assume a similar merge_n_bias function or just add the full bias here)
    merged_qkv = merged_qkv + in_proj_bias

    # 3. Manually Split Q, K, V
    q, k, v = torch.chunk(merged_qkv, 3, dim=-1)

    # 4. Reshape for Multi-Head: [Batch, Tokens, Dim] -> [Batch, Heads, Tokens, Head_Dim]

    q, k, v = ready_for_math(q, meta), ready_for_math(k, meta), ready_for_math(v, meta)

    # 5. The "Global" Attention Calculation (Softmax(QK^T / sqrt(dk)) * V)
    # This is where the patches actually "talk"
    scaling = head_dim ** -0.5
    attn_scores = (q @ k.transpose(-2, -1)) * scaling
    attn_probs = torch.nn.functional.softmax(attn_scores, dim=-1)
    context_layer = attn_probs @ v # [1, 12, 197, 64]

    # 6. Re-combine Heads and Project Out
    # [1, 12, 197, 64] -> [1, 197, 768]
    context_layer = context_layer.transpose(1, 2).reshape(1, meta["seq_length"], meta["embed_dim"])
    attn_output = block.self_attention.out_proj(context_layer)

    # 7. Residual Connection
    current_state = identity + attn_output
    
    # --- STEP 2: RESIDUAL 2 (MLP) ---
    identity = current_state
    x = block.ln_2(current_state)
    
    # Calculate MLP Neuron Slices
    mlp_split = int(meta["mlp_hidden_dim"] * RATIO)
    
    # Split Weights for Linear1 and Linear2
    w1, w2 = block.mlp[0].weight, block.mlp[3].weight
    b1, b2 = block.mlp[0].bias, block.mlp[3].bias
    
    # Edge MLP (Partial Hidden Layer)
    edge_h = torch.nn.functional.gelu(x @ w1[:mlp_split, :].t() + b1[:mlp_split])
    edge_out = edge_h @ w2[:, :mlp_split].t()
    
    # GPU MLP (Partial Hidden Layer)
    gpu_h = torch.nn.functional.gelu(x @ w1[mlp_split:, :].t() + b1[mlp_split:])
    gpu_out = gpu_h @ w2[:, mlp_split:].t()
    
    # Final MLP Merge (Summation + shared bias)
    mlp_final = (edge_out + gpu_out) + b2
    current_state = identity + mlp_final

    print(f"Finished Block {i} | Tokens: {meta['seq_length']} | Dims: {meta['embed_dim']}")

# Final Result
print("\nInference Loop Finished.")

Finished Block 0 | Tokens: 197 | Dims: 768
Finished Block 1 | Tokens: 197 | Dims: 768
Finished Block 2 | Tokens: 197 | Dims: 768
Finished Block 3 | Tokens: 197 | Dims: 768
Finished Block 4 | Tokens: 197 | Dims: 768
Finished Block 5 | Tokens: 197 | Dims: 768
Finished Block 6 | Tokens: 197 | Dims: 768
Finished Block 7 | Tokens: 197 | Dims: 768
Finished Block 8 | Tokens: 197 | Dims: 768
Finished Block 9 | Tokens: 197 | Dims: 768
Finished Block 10 | Tokens: 197 | Dims: 768
Finished Block 11 | Tokens: 197 | Dims: 768

Inference Loop Finished.


### 1. Attention: The "Side-by-Side" (Concat) Logic
In Multi-Head Self-Attention (MSA), each "head" is a completely independent observer. One head might look at color gradients, another at vertical lines, and another at the relationship between the top-left and bottom-right of the image.

* **The Structure:** These heads are mathematically placed **side-by-side**. 
* **The Split:** When you assign 3 heads to the Edge and 9 to the GPU, they are calculating entirely different "features" of the same image.
* **The Merge:** Because they are side-by-side, you cannot add them. Adding them would be like trying to average a "color" feature with a "shape" feature—it would result in gibberish. You must **concatenate** them to rebuild the full "feature vector" (the 768 dimensions).



**Documentation Key:** Concat = Rebuilding a wide vector from independent feature slices.

---

### 2. MLP: The "Partial Contribution" (Sum) Logic
The MLP (Multi-Layer Perceptron) works differently because of the way matrix multiplication is distributed. Think of the final output of an MLP as a **giant sum**.

Every single neuron in the hidden layer (the 3,072 neurons) looks at the input and says: *"Based on what I see, I think the final output should be adjusted by exactly this much."*

* **The Structure:** The final output is the **Sum** of all 3,072 neurons' opinions.
* **The Split:** * The **Edge** calculates the opinions of Neurons $1 \dots 1000$ and sums them up into a "Partial Result" vector of size 768.
    * The **GPU** calculates the opinions of Neurons $1001 \dots 3072$ and sums them up into a "Partial Result" vector of size 768.
* **The Merge:** Since both results are already in the final "768-dimension" space, you simply **Add** them. 
    * `Final_Output = (Edge_Partial_Sum) + (GPU_Partial_Sum)`
    * Mathematically, this is identical to calculating all 3,072 at once because of the **distributive property** of matrix multiplication: $A(x+y) = Ax + Ay$.



---

### 3. Summary Comparison

| Feature | MSA (Attention) | MLP (Feed-Forward) |
| :--- | :--- | :--- |
| **Independence** | Heads are independent *features*. | Neurons are independent *contributors*. |
| **Result Shape** | Each device produces a **thin** tensor. | Each device produces a **full-width** tensor. |
| **Merge Action** | `torch.cat([A, B], dim=-1)` | `A + B` |
| **Communication** | Must send the "width" of the assigned heads. | Must always send the "full width" (768). |

### Why this matters for your implementation:
This actually means that **MLP communication is more expensive** than Attention communication. 
* If you offload 1 head of Attention, you only send $1/12$th of the data back ($64$ floats per token).
* If you offload 1 neuron of the MLP, you still have to send the **entire 768-dim vector** back to the Master so it can be added.

# ``FINAL CLASSIFICATION``

In [52]:
# --- STEP 3: FINAL CLASSIFICATION ---

# 1. Final Layer Normalization
# The encoder has a standalone norm layer after all blocks are done
final_norm = model.encoder.ln(current_state)

# 2. Extract the [CLS] token
# In ViT, the first token is the 'summary' of the image
cls_token = final_norm[:, 0, :] # Shape: [1, 768]

# 3. The Classifier Head
# We'll simulate a split here too, just to be consistent with SPViT logic
# Split the 1000 classes: Edge does 400, GPU does 600
class_split = int(1000 * RATIO)

# Extract weights from model.head (The Linear layer)
head_w = model.heads.head.weight # [1000, 768]
head_b = model.heads.head.bias   # [1000]

# Edge calculates first 400 classes
edge_logits = cls_token @ head_w[:class_split, :].t() + head_b[:class_split]

# GPU calculates remaining 600 classes
gpu_logits = cls_token @ head_w[class_split:, :].t() + head_b[class_split:]

# 4. Final Merge (Concatenate classes)
# Since classes are independent "bins", we CONCAT them
final_logits = torch.cat([edge_logits, gpu_logits], dim=-1)

# 5. Get the Result
probabilities = torch.nn.functional.softmax(final_logits, dim=-1)
top_class = torch.argmax(probabilities, dim=-1)

print(f"Final Classification Logits Shape: {final_logits.shape}")
print(f"Predicted Class ID: {top_class.item()}")

Final Classification Logits Shape: torch.Size([1, 1000])
Predicted Class ID: 99


# To Toggle the ``RATIO`` we need the ARIMA-V

In [ ]:
import numpy as np
import time

class MultiDeviceARIMAManager:
    def __init__(self, device_ids, p=3, d=1, q=1):
        """
        device_ids: List of strings/names (e.g., ['edge', 'mid_pc', 'gpu'])
        p: Memory (how many past blocks to look at)
        d: Trend (detecting if a device is getting slower/hotter)
        q: Shock Absorber (correcting for sudden network lag)
        """
        self.p, self.d, self.q = p, d, q
        self.devices = device_ids
        
        # We keep a separate history for every single device
        self.history = {dev: [] for dev in device_ids}
        self.residuals = {dev: [] for dev in device_ids}
        self.last_predictions = {dev: 0.0 for dev in device_ids}
        
        # Start with an even split: 1.0 / number of devices
        initial_share = 1.0 / len(device_ids)
        self.current_shares = {dev: initial_share for dev in device_ids}

    def record_block_latency(self, device_id, latency):
        """
        Updates the history for a specific device after a block finishes.
        latency: Time in seconds (Ts + Tc) 
        Where Ts is Transmission Time and Tc is computaion time
        """
        if self.last_predictions[device_id] > 0:
            error = latency - self.last_predictions[device_id]
            self.residuals[device_id].append(error)
        
        self.history[device_id].append(latency)
        
        # Keep buffer small for performance on your i5 ProBook
        if len(self.history[device_id]) > 20:  # keep only history of last 20 block computations
            self.history[device_id].pop(0)

    def _predict_device_latency(self, dev):
        """Internal ARIMA math: Predicts the next latency for one device."""
        h = self.history[dev]
        if len(h) < self.p + self.d:
            return np.mean(h) if h else 0.1 # Default 100ms guess
        
        # Differencing (d=1): Looking at the change between blocks
        diffs = [h[i] - h[i-1] for i in range(1, len(h))]
        
        # AR Part (p): Weighted average of recent changes
        ar_val = np.mean(diffs[-self.p:]) # take mean of last p blocks latencies difference to predict the next one
        
        # MA Part (q): Adjustment based on previous error
        ma_val = 0.1 * self.residuals[dev][-1] if self.residuals[dev] else 0
        
        prediction = h[-1] + ar_val + ma_val
        self.last_predictions[dev] = max(0.001, prediction) # Latency can't be 0
        return self.last_predictions[dev]

    def update_shares(self):
        """
        The 'Orchestrator' logic. 
        Calculates the new % of heads/neurons each device should get.
        """
        predictions = {dev: self._predict_device_latency(dev) for dev in self.devices}
        
        # Inverse Propotional Logic: 
        # A device that is TWICE as slow should get HALF the work.
        # Score = 1 / Predicted_Latency
        scores = {dev: 1.0 / pred for dev, pred in predictions.items()} # more slow, less work (1 / pred) where pred is latency prediction
        total_score = sum(scores.values())                              # so the score can be converted into percentages that'll be used for RATIO
        
        # Normalize scores so they sum to 1.0 (100% of the model)
        for dev in self.devices:
            self.current_shares[dev] = scores[dev] / total_score
            
        return self.current_shares

    def get_head_indices(self, dev, total_heads):
        """
        Translates the % share into actual head indices for your slicing code.
        Example: If 'edge' has 0.25 share of 12 heads, it returns range(0, 3).
        """
        start_idx = 0
        for d in self.devices:
            share = self.current_shares[d]
            count = int(round(share * total_heads))
            
            # Ensure at least 1 head per device to keep the socket alive
            count = max(1, count) 
            
            end_idx = start_idx + count
            if d == dev:
                # Clip to total_heads to avoid index out of bounds
                return range(start_idx, min(end_idx, total_heads))
            start_idx = end_idx
        return range(0, 1) # Fallback

## Example Code change with ARIMA-V

In [ ]:
# 1. Setup the Manager for your devices
# Let's assume you have your i5 ProBook (edge) and a Desktop (gpu)
device_list = ['edge', 'gpu']
arima_manager = MultiDeviceARIMAManager(device_list)

# 2. Start the Inference
current_state = X 

for i, block in enumerate(model.encoder.layers):
    # --- DYNAMIC UPDATE STEP ---
    # Ask ARIMA to predict the best split based on previous block performance
    shares = arima_manager.update_shares()
    
    # Get specific head indices for each device
    edge_indices = arima_manager.get_head_indices('edge', meta["num_heads"])
    gpu_indices = arima_manager.get_head_indices('gpu', meta["num_heads"])
    
    # Record start time to measure this block's actual performance
    start_time = time.time()
    
    # --- MATH STEP (Using the dynamic indices) ---
    # Edge Attention
    edge_attn_w = get_head_weights(block.self_attention.in_proj_weight, edge_indices, ...)
    edge_qkv = x @ edge_attn_w.t()
    
    # GPU Attention (In a real setup, this would be a socket call)
    gpu_attn_w = get_head_weights(block.self_attention.in_proj_weight, gpu_indices, ...)
    gpu_qkv = x @ gpu_attn_w.t()
    
    # --- FEEDBACK STEP ---
    # Measure the total time it took for the "slowest" device to finish + network
    total_block_time = time.time() - start_time
    
    # Update the manager so it learns for Block i+1
    # For simulation, we attribute the time to both. In real life, 
    # you'd measure the GPU's return trip specifically.
    arima_manager.record_block_latency('edge', total_block_time)
    arima_manager.record_block_latency('gpu', total_block_time)
    
    print(f"Block {i} Split -> Edge: {len(edge_indices)} heads, GPU: {len(gpu_indices)} heads")